In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# STEP 2: Unzip (change path if needed)
import zipfile
import os
zip_path = '/content/drive/MyDrive/primary_dataset-1.zip' # Update if zip file of dataset is stored elsewhere
extract_path = '/content/bd_money'
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)
print("✅ Dataset unzipped to:", extract_path)

In [ ]:
!pip install lime

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import pandas as pd
from collections import defaultdict
import random

from sklearn.model_selection import KFold, train_test_split
from sklearn.preprocessing import LabelEncoder, Normalizer
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, cohen_kappa_score, matthews_corrcoef,
    roc_auc_score, roc_curve, auc
)

# LIME
import lime
from lime import lime_image
from lime.wrappers.scikit_image import SegmentationAlgorithm

# TensorFlow / Keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV3Large, EfficientNetB0
from tensorflow.keras.applications.mobilenet_v3 import preprocess_input as mobilenet_preprocess
from tensorflow.keras.applications.efficientnet import preprocess_input as efficientnet_preprocess

# Set random seeds for reproducibility
np.random.seed(42)
random.seed(42)

# ==============================
# Configuration
# ==============================
CONFIG = {
    'train_dir': '/content/bd_money/primary_dataset-1/train',
    'test_dir': '/content/bd_money/primary_dataset-1/test',
    'n_folds': 5,
    'lime_samples': 1000,
    'lime_examples_per_fold': 1,
    'input_shape': (224, 224, 3),
    'random_state': 42
}

# ==============================
# Data Augmentor
# ==============================
augmentor = ImageDataGenerator(
    rotation_range=30,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=[0.8, 1.2],
    brightness_range=[0.8, 1.2],
    horizontal_flip=True,
    fill_mode='nearest',
)

# ==============================
# Initialize Pre-trained Models
# ==============================
print("Loading pre-trained models...")
mobilenet_model = MobileNetV3Large(
    weights='imagenet',
    include_top=False,
    pooling='avg',
    input_shape=CONFIG['input_shape']
)
efficientnet_model = EfficientNetB0(
    weights='imagenet',
    include_top=False,
    pooling='avg',
    input_shape=CONFIG['input_shape']
)

# Freeze most layers for fine-tuning
for layer in mobilenet_model.layers[:-20]:
    layer.trainable = False
for layer in efficientnet_model.layers[:-20]:
    layer.trainable = False

print("Pre-trained models loaded successfully!")

# ==============================
# Feature Extraction Functions
# ==============================
def extract_features_from_directory(directory_path, augment_data=True):
    """Extract features from all images in directory"""
    features, labels, image_paths = [], [], []

    if not os.path.exists(directory_path):
        raise FileNotFoundError(f"Directory not found: {directory_path}")

    class_folders = [f for f in os.listdir(directory_path) if os.path.isdir(os.path.join(directory_path, f))]

    for label in tqdm(class_folders, desc="Extracting features"):
        class_dir = os.path.join(directory_path, label)

        for img_name in os.listdir(class_dir):
            img_path = os.path.join(class_dir, img_name)
            img = cv2.imread(img_path)
            if img is None:
                continue

            try:
                orig = cv2.resize(img, CONFIG['input_shape'][:2])
                features_orig = extract_single_image_features(orig)
                features.append(features_orig)
                labels.append(label)
                image_paths.append(img_path)

                if augment_data:
                    aug = augmentor.random_transform(orig)
                    features_aug = extract_single_image_features(aug)
                    features.append(features_aug)
                    labels.append(label)
                    image_paths.append(img_path + "_aug")

            except Exception as e:
                print(f"Error processing {img_path}: {e}")

    return np.array(features), np.array(labels), image_paths

def extract_single_image_features(image):
    """Extract features from a single image"""
    mob_feat = mobilenet_model.predict(
        np.expand_dims(mobilenet_preprocess(image.copy()), axis=0),
        verbose=0
    )
    eff_feat = efficientnet_model.predict(
        np.expand_dims(efficientnet_preprocess(image.copy()), axis=0),
        verbose=0
    )
    return np.concatenate((mob_feat.flatten(), eff_feat.flatten()))

# ==============================
# Extract Training Features
# ==============================
print("Extracting training features...")
X, y, train_paths = extract_features_from_directory(CONFIG['train_dir'], augment_data=True)

# Label encoding
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# L2 normalization
l2_normalizer = Normalizer(norm='l2')
X_normalized = l2_normalizer.fit_transform(X)

# PCA (retain 95% variance)
pca = PCA(n_components=0.95, random_state=CONFIG['random_state'])
X_pca = pca.fit_transform(X_normalized)

print(f"Original feature dimensions: {X.shape}")
print(f"PCA reduced dimensions: {X_pca.shape}")
print(f"Number of classes: {len(label_encoder.classes_)}")
print(f"Classes: {label_encoder.classes_}")

In [ ]:
# ==============================
# Model Architecture
# ==============================
def create_model(input_dim, num_classes):
    """Create neural network model"""
    model = Sequential([
        Dense(512, activation='relu', input_shape=(input_dim,)),
        BatchNormalization(),
        Dropout(0.5),

        Dense(256, activation='relu'),
        BatchNormalization(),
        Dropout(0.5),

        Dense(128, activation='relu'),
        BatchNormalization(),
        Dropout(0.4),

        Dense(64, activation='relu'),
        BatchNormalization(),
        Dropout(0.3),

        Dense(num_classes, activation='softmax')
    ])
    model.compile(
        optimizer=Adam(learning_rate=1e-4, weight_decay=1e-4),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

# ==============================
# Enhanced Metrics Calculation with AUC and MCC
# ==============================
def calculate_comprehensive_metrics(y_true, y_pred, y_pred_proba=None):
    """Calculate all metrics including Cohen's Kappa, MCC, and AUC"""
    metrics = {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, average='weighted', zero_division=0),
        "Recall": recall_score(y_true, y_pred, average='weighted', zero_division=0),
        "F1": f1_score(y_true, y_pred, average='weighted', zero_division=0),
        "Cohen_Kappa": cohen_kappa_score(y_true, y_pred),
        "MCC": matthews_corrcoef(y_true, y_pred)
    }

    # Add AUC for multiclass
    if y_pred_proba is not None:
        try:
            if len(np.unique(y_true)) > 2:
                metrics["AUC"] = roc_auc_score(y_true, y_pred_proba, multi_class='ovr', average='weighted')
            else:
                metrics["AUC"] = roc_auc_score(y_true, y_pred_proba[:, 1])
        except:
            metrics["AUC"] = 0.0
    else:
        metrics["AUC"] = 0.0

    return metrics

# ==============================
# Test Evaluation Function with Probabilities
# ==============================
def evaluate_on_test_set(model, test_dir, mobilenet_model, efficientnet_model,
                         l2_normalizer, pca, label_encoder):
    """Evaluate model on test set and return results with probabilities"""
    y_true, y_pred, y_pred_proba = [], [], []
    test_image_info = []

    test_files = [f for f in os.listdir(test_dir) if "_" in f]

    for img_name in tqdm(test_files, desc="Testing"):
        true_label = img_name.split("_")[0]
        img_path = os.path.join(test_dir, img_name)
        img = cv2.imread(img_path)

        if img is None:
            continue

        try:
            img_resized = cv2.resize(img, CONFIG['input_shape'][:2])
            features = extract_single_image_features(img_resized)
            norm_feat = l2_normalizer.transform([features])
            reduced_feat = pca.transform(norm_feat)

            pred_probs = model.predict(reduced_feat, verbose=0)[0]
            pred_label = label_encoder.inverse_transform([np.argmax(pred_probs)])[0]

            y_pred.append(pred_label)
            y_true.append(true_label)
            y_pred_proba.append(pred_probs)

            test_image_info.append({
                'path': img_path,
                'name': img_name,
                'true_label': true_label,
                'pred_label': pred_label,
                'confidence': np.max(pred_probs)
            })

        except Exception as e:
            print(f"Error processing {img_name}: {e}")

    return y_true, y_pred, np.array(y_pred_proba), test_image_info

# ==============================
# All Folds Radar Plot with Maximum Values Only
# ==============================
def create_all_folds_radar_plot(all_fold_data):
    """Create radar plots showing all folds with only maximum values labeled"""

    radar_metrics = ['Accuracy', 'Precision', 'Recall', 'F1', 'Cohen_Kappa', 'MCC', 'AUC']

    # Calculate maximum values for each metric
    max_val_metrics = {}
    max_test_metrics = {}
    best_fold_val = {}
    best_fold_test = {}

    for metric in radar_metrics:
        max_val = -float('inf')
        max_test = -float('inf')
        best_val_fold = 1
        best_test_fold = 1

        for fold_data in all_fold_data:
            val_value = fold_data['val_metrics'][metric]
            test_value = fold_data['test_metrics'][metric]

            if val_value > max_val:
                max_val = val_value
                best_val_fold = fold_data['fold']

            if test_value > max_test:
                max_test = test_value
                best_test_fold = fold_data['fold']

        max_val_metrics[metric] = max_val
        max_test_metrics[metric] = max_test
        best_fold_val[metric] = best_val_fold
        best_fold_test[metric] = best_test_fold

    # Set up the radar plot
    angles = np.linspace(0, 2 * np.pi, len(radar_metrics), endpoint=False).tolist()
    angles += angles[:1]

    # Create color maps for folds
    val_colors = plt.cm.Blues(np.linspace(0.4, 0.9, CONFIG['n_folds']))
    test_colors = plt.cm.Reds(np.linspace(0.4, 0.9, CONFIG['n_folds']))

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(24, 12), subplot_kw=dict(projection='polar'))

    # Plot 1: All Validation Folds
    ax1.set_title('Validation Metrics - All Folds\n(Cohen\'s Kappa and MCC normalized to [0,1])',
                  fontsize=14, fontweight='bold', pad=30)

    for fold_data in all_fold_data:
        fold = fold_data['fold']
        val_values = []

        for metric in radar_metrics:
            original_val = fold_data['val_metrics'][metric]
            # Normalize Cohen's Kappa and MCC from [-1, 1] to [0, 1]
            if metric in ['Cohen_Kappa', 'MCC']:
                val_values.append((original_val + 1) / 2)
            else:
                val_values.append(original_val)

        val_values += val_values[:1]

        ax1.plot(angles, val_values, 'o-', linewidth=2, markersize=6,
                label=f'Fold {fold}', color=val_colors[fold-1], alpha=0.7)
        ax1.fill(angles, val_values, alpha=0.1, color=val_colors[fold-1])

    # Add labels only for maximum values
    val_original = []
    val_normalized = []
    for metric in radar_metrics:
        original_val = max_val_metrics[metric]
        val_original.append(original_val)
        if metric in ['Cohen_Kappa', 'MCC']:
            val_normalized.append((original_val + 1) / 2)
        else:
            val_normalized.append(original_val)

    for angle_idx, angle in enumerate(angles[:-1]):
        radius = val_normalized[angle_idx]
        original_val = val_original[angle_idx]
        fold_num = best_fold_val[radar_metrics[angle_idx]]

        label_radius = radius + 0.12
        ax1.text(angle, label_radius, f'{original_val:.3f}\n(F{fold_num})',
                ha='center', va='center', fontsize=9, fontweight='bold',
                color='navy', bbox=dict(boxstyle="round,pad=0.3",
                facecolor='white', alpha=0.9, edgecolor='navy', linewidth=2))

    ax1.set_xticks(angles[:-1])
    ax1.set_xticklabels(radar_metrics, fontsize=13, fontweight='bold')
    ax1.set_ylim(0, 1.3)
    ax1.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
    ax1.set_yticklabels(['0.2', '0.4', '0.6', '0.8', '1.0'], fontsize=11)
    ax1.grid(True, alpha=0.3, linewidth=1.5)
    ax1.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=10)

    # Plot 2: All Test Folds
    ax2.set_title('Test Metrics - All Folds\n(Cohen\'s Kappa and MCC normalized to [0,1])',
                  fontsize=14, fontweight='bold', pad=30)

    for fold_data in all_fold_data:
        fold = fold_data['fold']
        test_values = []

        for metric in radar_metrics:
            original_val = fold_data['test_metrics'][metric]
            # Normalize Cohen's Kappa and MCC from [-1, 1] to [0, 1]
            if metric in ['Cohen_Kappa', 'MCC']:
                test_values.append((original_val + 1) / 2)
            else:
                test_values.append(original_val)

        test_values += test_values[:1]

        ax2.plot(angles, test_values, 's-', linewidth=2, markersize=6,
                label=f'Fold {fold}', color=test_colors[fold-1], alpha=0.7)
        ax2.fill(angles, test_values, alpha=0.1, color=test_colors[fold-1])

    # Add labels only for maximum values
    test_original = []
    test_normalized = []
    for metric in radar_metrics:
        original_val = max_test_metrics[metric]
        test_original.append(original_val)
        if metric in ['Cohen_Kappa', 'MCC']:
            test_normalized.append((original_val + 1) / 2)
        else:
            test_normalized.append(original_val)

    for angle_idx, angle in enumerate(angles[:-1]):
        radius = test_normalized[angle_idx]
        original_val = test_original[angle_idx]
        fold_num = best_fold_test[radar_metrics[angle_idx]]

        label_radius = radius + 0.12
        ax2.text(angle, label_radius, f'{original_val:.3f}\n(F{fold_num})',
                ha='center', va='center', fontsize=9, fontweight='bold',
                color='darkred', bbox=dict(boxstyle="round,pad=0.3",
                facecolor='white', alpha=0.9, edgecolor='darkred', linewidth=2))

    ax2.set_xticks(angles[:-1])
    ax2.set_xticklabels(radar_metrics, fontsize=13, fontweight='bold')
    ax2.set_ylim(0, 1.3)
    ax2.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
    ax2.set_yticklabels(['0.2', '0.4', '0.6', '0.8', '1.0'], fontsize=11)
    ax2.grid(True, alpha=0.3, linewidth=1.5)
    ax2.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=10)

    plt.suptitle(f'Performance Metrics - All {CONFIG["n_folds"]} Folds\n(Only maximum values labeled with fold number)',
                 fontsize=16, fontweight='bold', y=0.98)
    plt.tight_layout()
    plt.show()

    # Print summary
    print("\n" + "="*80)
    print("MAXIMUM VALUES SUMMARY")
    print("="*80)
    print("\nValidation Metrics (Maximum Values):")
    for metric in radar_metrics:
        print(f"  {metric}: {max_val_metrics[metric]:.4f} (Fold {best_fold_val[metric]})")

    print("\nTest Metrics (Maximum Values):")
    for metric in radar_metrics:
        print(f"  {metric}: {max_test_metrics[metric]:.4f} (Fold {best_fold_test[metric]})")

# # ==============================
# # LIME Functions
# # ==============================
def predict_pipeline_for_lime(images, model, mobilenet_model, efficientnet_model,
                              l2_normalizer, pca, label_encoder):
    """Prediction pipeline for LIME explanations"""
    predictions = []
    for img in images:
        try:
            if len(img.shape) == 3 and img.shape[2] == 3:
                img_bgr = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
            else:
                img_bgr = img

            img_resized = cv2.resize(img_bgr, CONFIG['input_shape'][:2])
            features = extract_single_image_features(img_resized)
            norm_feat = l2_normalizer.transform([features])
            reduced_feat = pca.transform(norm_feat)
            pred_probs = model.predict(reduced_feat, verbose=0)[0]
            predictions.append(pred_probs)

        except Exception as e:
            uniform_probs = np.ones(len(label_encoder.classes_)) / len(label_encoder.classes_)
            predictions.append(uniform_probs)

    return np.array(predictions)

def generate_lime_explanation(model, test_image_path, mobilenet_model, efficientnet_model,
                             l2_normalizer, pca, label_encoder, num_samples=1000):
    """Generate LIME explanation for a single image"""
    try:
        img = cv2.imread(test_image_path)
        if img is None:
            return None, None, None

        img = cv2.resize(img, CONFIG['input_shape'][:2])
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        explainer = lime_image.LimeImageExplainer()

        def predict_fn(images):
            return predict_pipeline_for_lime(
                images, model, mobilenet_model, efficientnet_model,
                l2_normalizer, pca, label_encoder
            )

        explanation = explainer.explain_instance(
            img_rgb,
            predict_fn,
            top_labels=min(5, len(label_encoder.classes_)),
            hide_color=0,
            num_samples=num_samples,
            segmentation_fn=SegmentationAlgorithm(
                'quickshift',
                kernel_size=4,
                max_dist=200,
                ratio=0.2
            )
        )

        return explanation, img_rgb, img

    except Exception as e:
        print(f"LIME error for {test_image_path}: {e}")
        return None, None, None

def visualize_lime_explanation(explanation, original_img, img_name, true_label, pred_label, fold, top_n=3):
    """Visualize LIME explanation"""
    try:
        fig, axes = plt.subplots(2, top_n + 1, figsize=(20, 10))

        axes[0, 0].imshow(original_img)
        axes[0, 0].set_title(f'Original Image\nTrue: {true_label}\nPred: {pred_label}', fontsize=12)
        axes[0, 0].axis('off')
        axes[1, 0].axis('off')

        top_labels = explanation.top_labels[:top_n]

        for i, label_idx in enumerate(top_labels):
            class_name = label_encoder.inverse_transform([label_idx])[0]

            temp_pos, _ = explanation.get_image_and_mask(
                label_idx, positive_only=True, num_features=10, hide_rest=False
            )
            axes[0, i + 1].imshow(temp_pos)
            axes[0, i + 1].set_title(f'Supporting Features\nClass: {class_name}', fontsize=10)
            axes[0, i + 1].axis('off')

            temp_all, _ = explanation.get_image_and_mask(
                label_idx, positive_only=False, num_features=10, hide_rest=False
            )
            axes[1, i + 1].imshow(temp_all)
            axes[1, i + 1].set_title(f'All Features\nClass: {class_name}', fontsize=10)
            axes[1, i + 1].axis('off')

        plt.suptitle(f'LIME Explanation - Fold {fold} - {img_name}', fontsize=14)
        plt.tight_layout()
        plt.show()

    except Exception as e:
        print(f"LIME visualization error: {e}")


# ==============================
# K-Fold Cross Validation with Enhanced Per-Fold Metrics
# ==============================
print("\nStarting K-Fold Cross Validation with Enhanced Metrics...")

kf = KFold(n_splits=CONFIG['n_folds'], shuffle=True, random_state=CONFIG['random_state'])
results = []
all_fold_metrics = []
best_model = None
best_val_accuracy = 0

for fold, (train_idx, val_idx) in enumerate(kf.split(X_pca, y_encoded), 1):
    print(f"\n{'='*50}")
    print(f"FOLD {fold}/{CONFIG['n_folds']}")
    print(f"{'='*50}")

    # Split data
    X_train, X_val = X_pca[train_idx], X_pca[val_idx]
    y_train, y_val = y_encoded[train_idx], y_encoded[val_idx]

    # Create and train model
    model = create_model(X_train.shape[1], len(np.unique(y_encoded)))

    callbacks = [
        EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-6, verbose=1)
    ]

    print("Training model...")
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=50,
        batch_size=32,
        verbose=1,
        callbacks=callbacks
    )

    # ==============================
    # TRAINING ACCURACY EVALUATION
    # ==============================
    print("\nEvaluating training accuracy...")
    train_preds_proba = model.predict(X_train, verbose=0)
    train_pred_labels = np.argmax(train_preds_proba, axis=1)
    train_metrics = calculate_comprehensive_metrics(y_train, train_pred_labels, train_preds_proba)

    print(f"Training Accuracy: {train_metrics['Accuracy']:.4f}")

    # Validation evaluation with enhanced metrics
    val_preds_proba = model.predict(X_val, verbose=0)
    val_pred_labels = np.argmax(val_preds_proba, axis=1)
    val_metrics = calculate_comprehensive_metrics(y_val, val_pred_labels, val_preds_proba)

    # Test evaluation with enhanced metrics
    print("Evaluating on test set...")
    y_true, y_pred, y_pred_proba, test_image_info = evaluate_on_test_set(
        model, CONFIG['test_dir'], mobilenet_model, efficientnet_model,
        l2_normalizer, pca, label_encoder
    )

    # Convert string labels to numeric for metrics calculation
    y_true_encoded = label_encoder.transform(y_true)
    y_pred_encoded = label_encoder.transform(y_pred)
    test_metrics = calculate_comprehensive_metrics(y_true_encoded, y_pred_encoded, y_pred_proba)

    # Print fold results
    print(f"\n{'='*50}")
    print(f"FOLD {fold} RESULTS SUMMARY")
    print(f"{'='*50}")

    print("\nTraining Metrics:")
    for metric, value in train_metrics.items():
        print(f"  {metric}: {value:.4f}")

    print("\nValidation Metrics:")
    for metric, value in val_metrics.items():
        print(f"  {metric}: {value:.4f}")

    print("\nTest Metrics:")
    for metric, value in test_metrics.items():
        print(f"  {metric}: {value:.4f}")

    # Store results with training metrics
    fold_results = {"Fold": fold}
    fold_results.update({f"Train_{k}": v for k, v in train_metrics.items()})
    fold_results.update({f"Val_{k}": v for k, v in val_metrics.items()})
    fold_results.update({f"Test_{k}": v for k, v in test_metrics.items()})
    results.append(fold_results)

    # Store individual fold metrics for analysis
    all_fold_metrics.append({
        'fold': fold,
        'train_metrics': train_metrics,
        'val_metrics': val_metrics,
        'test_metrics': test_metrics
    })

    # Keep track of best model
    if val_metrics['Accuracy'] > best_val_accuracy:
        best_val_accuracy = val_metrics['Accuracy']
        best_model = model
        best_fold = fold
        print(f"\n  → New best model! (Validation Accuracy: {best_val_accuracy:.4f})")

    print(f"\nFold {fold} completed!")

# ==============================
# Comprehensive Results Summary
# ==============================
print("\n" + "="*80)
print("COMPREHENSIVE CROSS-VALIDATION RESULTS SUMMARY")
print("="*80)

df_results = pd.DataFrame(results)
print("\nComplete Results Across All Folds:")
print(df_results.round(4))

print("\n" + "="*80)
print("AVERAGE METRICS ACROSS ALL FOLDS")
print("="*80)
avg_metrics = df_results.mean(numeric_only=True)

print("\nTraining Metrics (Average):")
for col in df_results.columns:
    if col.startswith('Train_'):
        metric_name = col.replace('Train_', '')
        print(f"  {metric_name}: {avg_metrics[col]:.4f}")

print("\nValidation Metrics (Average):")
for col in df_results.columns:
    if col.startswith('Val_'):
        metric_name = col.replace('Val_', '')
        print(f"  {metric_name}: {avg_metrics[col]:.4f}")

print("\nTest Metrics (Average):")
for col in df_results.columns:
    if col.startswith('Test_'):
        metric_name = col.replace('Test_', '')
        print(f"  {metric_name}: {avg_metrics[col]:.4f}")

# ==============================
# Generate All Folds Radar Plot
# ==============================
print("\n" + "="*60)
print("GENERATING ALL FOLDS RADAR PLOT")
print("="*60)
print("Creating radar plot showing all folds with maximum values labeled...")

create_all_folds_radar_plot(all_fold_metrics)

# ==============================
# Per-Metric Analysis Across Folds
# ==============================
print("\n" + "="*60)
print("PER-METRIC ANALYSIS ACROSS FOLDS")
print("="*60)

metrics_to_analyze = ['Accuracy', 'Precision', 'Recall', 'F1', 'Cohen_Kappa', 'MCC', 'AUC']

for metric in metrics_to_analyze:
    train_col = f'Train_{metric}'
    val_col = f'Val_{metric}'
    test_col = f'Test_{metric}'

    print(f"\n{metric}:")
    print(f"  Training - Mean: {df_results[train_col].mean():.4f}, Std: {df_results[train_col].std():.4f}, Max: {df_results[train_col].max():.4f}")
    print(f"  Validation - Mean: {df_results[val_col].mean():.4f}, Std: {df_results[val_col].std():.4f}, Max: {df_results[val_col].max():.4f}")
    print(f"  Test - Mean: {df_results[test_col].mean():.4f}, Std: {df_results[test_col].std():.4f}, Max: {df_results[test_col].max():.4f}")

# ==============================
# Comprehensive Visualization (Extended)
# ==============================
print("\nGenerating comprehensive visualization with training metrics...")

fig, axes = plt.subplots(3, 3, figsize=(24, 15))
axes = axes.ravel()

for idx, metric in enumerate(metrics_to_analyze):
    ax = axes[idx]

    train_values = df_results[f'Train_{metric}'].values
    val_values = df_results[f'Val_{metric}'].values
    test_values = df_results[f'Test_{metric}'].values

    ax.plot(df_results['Fold'], train_values, 'go-', linewidth=2, markersize=8, label='Training')
    ax.plot(df_results['Fold'], val_values, 'bo-', linewidth=2, markersize=8, label='Validation')
    ax.plot(df_results['Fold'], test_values, 'ro-', linewidth=2, markersize=8, label='Test')

    # Highlight maximum values
    max_train_idx = np.argmax(train_values)
    max_val_idx = np.argmax(val_values)
    max_test_idx = np.argmax(test_values)

    ax.plot(df_results['Fold'].iloc[max_train_idx], train_values[max_train_idx], 'g*',
            markersize=20, label=f'Max Train: {train_values[max_train_idx]:.3f}')
    ax.plot(df_results['Fold'].iloc[max_val_idx], val_values[max_val_idx], 'b*',
            markersize=20, label=f'Max Val: {val_values[max_val_idx]:.3f}')
    ax.plot(df_results['Fold'].iloc[max_test_idx], test_values[max_test_idx], 'r*',
            markersize=20, label=f'Max Test: {test_values[max_test_idx]:.3f}')

    ax.set_title(f'{metric} Across Folds', fontweight='bold', fontsize=12)
    ax.set_xlabel('Fold', fontsize=10)
    ax.set_ylabel(metric, fontsize=10)
    ax.legend(fontsize=8, loc='best')
    ax.grid(True, alpha=0.3)

    # Set y-limits based on metric type
    if metric in ['Cohen_Kappa', 'MCC']:
        ax.set_ylim(-1, 1)
    else:
        ax.set_ylim(0, 1)

# Overall average comparison
ax = axes[-2]
train_means = [avg_metrics[f'Train_{metric}'] for metric in metrics_to_analyze]
val_means = [avg_metrics[f'Val_{metric}'] for metric in metrics_to_analyze]
test_means = [avg_metrics[f'Test_{metric}'] for metric in metrics_to_analyze]

x_pos = np.arange(len(metrics_to_analyze))
width = 0.25

bars1 = ax.bar(x_pos - width, train_means, width, label='Training', alpha=0.8, color='green')
bars2 = ax.bar(x_pos, val_means, width, label='Validation', alpha=0.8, color='blue')
bars3 = ax.bar(x_pos + width, test_means, width, label='Test', alpha=0.8, color='red')

ax.set_xlabel('Metrics', fontsize=10)
ax.set_ylabel('Average Score', fontsize=10)
ax.set_title('Average Performance Across All Folds', fontweight='bold', fontsize=12)
ax.set_xticks(x_pos)
ax.set_xticklabels(metrics_to_analyze, rotation=45, ha='right')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                f'{height:.3f}', ha='center', va='bottom', fontsize=7)

# Hide the last unused subplot
axes[-1].axis('off')

plt.suptitle('Enhanced Cross-Validation Analysis - Training, Validation & Test Metrics',
             fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

# ==============================
# Save Results
# ==============================
try:
    # Save detailed results
    df_results.to_csv('enhanced_per_fold_results_with_training.csv', index=False)

    # Save summary statistics including max values
    summary_stats = {}
    for metric in metrics_to_analyze:
        train_col = f'Train_{metric}'
        val_col = f'Val_{metric}'
        test_col = f'Test_{metric}'

        summary_stats[f'{metric}_Train_Mean'] = df_results[train_col].mean()
        summary_stats[f'{metric}_Train_Std'] = df_results[train_col].std()
        summary_stats[f'{metric}_Train_Max'] = df_results[train_col].max()

        summary_stats[f'{metric}_Val_Mean'] = df_results[val_col].mean()
        summary_stats[f'{metric}_Val_Std'] = df_results[val_col].std()
        summary_stats[f'{metric}_Val_Max'] = df_results[val_col].max()

        summary_stats[f'{metric}_Test_Mean'] = df_results[test_col].mean()
        summary_stats[f'{metric}_Test_Std'] = df_results[test_col].std()
        summary_stats[f'{metric}_Test_Max'] = df_results[test_col].max()

    pd.DataFrame([summary_stats]).to_csv('metrics_summary_statistics_with_training.csv', index=False)

    print("\nResults saved:")
    print("  • enhanced_per_fold_results_with_training.csv - Detailed per-fold results with training metrics")
    print("  • metrics_summary_statistics_with_training.csv - Summary statistics including training metrics")

except Exception as e:
    print(f"\nError saving results: {e}")

print(f"\n{'='*80}")
print("ANALYSIS COMPLETE!")
print(f"{'='*80}")
print(f"✓ Enhanced Per-Fold Analysis with Training, Validation, and Test Metrics")
print(f"✓ Training Accuracy printed for each fold")
print(f"✓ Best Model: Fold {best_fold} (Validation Accuracy: {best_val_accuracy:.4f})")
print(f"✓ All visualizations and results saved")
print(f"{'='*80}")

In [ ]:
# ==============================
# FINAL MODEL TRAINING & COMPREHENSIVE EVALUATION
# ==============================
# This cell trains the final model and provides complete evaluation
# Run this AFTER the cross-validation analysis is complete

import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import pandas as pd
import random
from itertools import cycle

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, cohen_kappa_score, matthews_corrcoef,
    roc_auc_score, roc_curve, auc, classification_report
)
from sklearn.preprocessing import label_binarize

import lime
from lime import lime_image
from lime.wrappers.scikit_image import SegmentationAlgorithm
import shap

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

print("="*80)
print("FINAL MODEL TRAINING & COMPREHENSIVE EVALUATION")
print("="*80)

# ==============================
# Configuration
# ==============================
FINAL_CONFIG = {
    'test_size': 0.2,
    'random_state': 42,
    'epochs': 100,
    'batch_size': 32,
    'learning_rate': 1e-3,
    'n_predictions_to_show': 15,
    'lime_explanations': 5,
    'shap_samples': 300
}

# ==============================
# Split Data into Train and Validation
# ==============================
print("\n[1/11] Splitting data into train and validation sets...")
X_train_final, X_val_final, y_train_final, y_val_final = train_test_split(
    X_pca, y_encoded,
    test_size=FINAL_CONFIG['test_size'],
    random_state=FINAL_CONFIG['random_state'],
    stratify=y_encoded
)

print(f"Training samples: {len(X_train_final)}")
print(f"Validation samples: {len(X_val_final)}")
print(f"Number of classes: {len(label_encoder.classes_)}")

# ==============================
# Create and Train Final Model
# ==============================
print("\n[2/11] Creating and training final model...")
final_model = create_model(X_train_final.shape[1], len(label_encoder.classes_))

callbacks_final = [
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-7, verbose=1)
]

history_final = final_model.fit(
    X_train_final, y_train_final,
    validation_data=(X_val_final, y_val_final),
    epochs=FINAL_CONFIG['epochs'],
    batch_size=FINAL_CONFIG['batch_size'],
    verbose=1,
    callbacks=callbacks_final
)

print("✓ Model training completed!")

# ==============================
# Training History Visualization
# ==============================
print("\n[3/11] Visualizing training history...")

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Accuracy plot
axes[0].plot(history_final.history['accuracy'], label='Training Accuracy', linewidth=2.5, color='#2E86AB')
axes[0].plot(history_final.history['val_accuracy'], label='Validation Accuracy', linewidth=2.5, color='#A23B72')
axes[0].set_title('Model Accuracy Over Epochs', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Accuracy', fontsize=12)
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Loss plot
axes[1].plot(history_final.history['loss'], label='Training Loss', linewidth=2.5, color='#2E86AB')
axes[1].plot(history_final.history['val_loss'], label='Validation Loss', linewidth=2.5, color='#A23B72')
axes[1].set_title('Model Loss Over Epochs', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Loss', fontsize=12)
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.suptitle('Final Model Training Curves', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# ==============================
# Training Set Evaluation
# ==============================
print("\n[4/11] Evaluating on training set...")
train_preds_proba_final = final_model.predict(X_train_final, verbose=0)
train_preds_final = np.argmax(train_preds_proba_final, axis=1)

train_metrics_final = calculate_comprehensive_metrics(y_train_final, train_preds_final, train_preds_proba_final)

print("\n" + "="*60)
print("TRAINING SET METRICS")
print("="*60)
for metric, value in train_metrics_final.items():
    print(f"{metric:15s}: {value:.4f}")
print("="*60)

# ==============================
# Validation Set Evaluation
# ==============================
print("\n[5/11] Evaluating on validation set...")
val_preds_proba_final = final_model.predict(X_val_final, verbose=0)
val_preds_final = np.argmax(val_preds_proba_final, axis=1)

val_metrics_final = calculate_comprehensive_metrics(y_val_final, val_preds_final, val_preds_proba_final)

print("\n" + "="*60)
print("VALIDATION SET METRICS")
print("="*60)
for metric, value in val_metrics_final.items():
    print(f"{metric:15s}: {value:.4f}")
print("="*60)

# ==============================
# Test Set Evaluation
# ==============================
print("\n[6/11] Evaluating on test set...")
y_true_test, y_pred_test, y_pred_proba_test, test_image_info_final = evaluate_on_test_set(
    final_model, CONFIG['test_dir'], mobilenet_model, efficientnet_model,
    l2_normalizer, pca, label_encoder
)

y_true_test_encoded = label_encoder.transform(y_true_test)
y_pred_test_encoded = label_encoder.transform(y_pred_test)

test_metrics_final = calculate_comprehensive_metrics(y_true_test_encoded, y_pred_test_encoded, y_pred_proba_test)

print("\n" + "="*60)
print("TEST SET METRICS")
print("="*60)
for metric, value in test_metrics_final.items():
    print(f"{metric:15s}: {value:.4f}")
print("="*60)

# ==============================
# Metrics Comparison Visualization
# ==============================
print("\n[7/11] Visualizing metrics comparison across all sets...")

metrics_to_compare = ['Accuracy', 'Precision', 'Recall', 'F1', 'Cohen_Kappa', 'MCC', 'AUC']

train_values = [train_metrics_final[m] for m in metrics_to_compare]
val_values = [val_metrics_final[m] for m in metrics_to_compare]
test_values = [test_metrics_final[m] for m in metrics_to_compare]

x_pos = np.arange(len(metrics_to_compare))
width = 0.25

fig, ax = plt.subplots(figsize=(16, 8))

bars1 = ax.bar(x_pos - width, train_values, width, label='Training', alpha=0.8, color='#2ecc71')
bars2 = ax.bar(x_pos, val_values, width, label='Validation', alpha=0.8, color='#3498db')
bars3 = ax.bar(x_pos + width, test_values, width, label='Test', alpha=0.8, color='#e74c3c')

ax.set_xlabel('Metrics', fontsize=13, fontweight='bold')
ax.set_ylabel('Score', fontsize=13, fontweight='bold')
ax.set_title('Performance Metrics Comparison - Training vs Validation vs Test',
             fontsize=15, fontweight='bold', pad=20)
ax.set_xticks(x_pos)
ax.set_xticklabels(metrics_to_compare, fontsize=11, fontweight='bold')
ax.legend(fontsize=12, loc='lower right')
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim([0, 1.1])

# Add value labels on bars
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                f'{height:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

# ==============================
# Confusion Matrices
# ==============================
print("\n[8/11] Generating confusion matrices...")

fig, axes = plt.subplots(1, 3, figsize=(24, 7))

# Training Confusion Matrix
cm_train = confusion_matrix(y_train_final, train_preds_final)
sns.heatmap(cm_train, annot=True, fmt='d', cmap='Greens',
            xticklabels=label_encoder.classes_,
            yticklabels=label_encoder.classes_,
            ax=axes[0], cbar_kws={'label': 'Count'})
axes[0].set_title(f'Training Set Confusion Matrix\nAccuracy: {train_metrics_final["Accuracy"]:.4f}',
                  fontsize=13, fontweight='bold')
axes[0].set_xlabel('Predicted Label', fontsize=11)
axes[0].set_ylabel('True Label', fontsize=11)

# Validation Confusion Matrix
cm_val = confusion_matrix(y_val_final, val_preds_final)
sns.heatmap(cm_val, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_encoder.classes_,
            yticklabels=label_encoder.classes_,
            ax=axes[1], cbar_kws={'label': 'Count'})
axes[1].set_title(f'Validation Set Confusion Matrix\nAccuracy: {val_metrics_final["Accuracy"]:.4f}',
                  fontsize=13, fontweight='bold')
axes[1].set_xlabel('Predicted Label', fontsize=11)
axes[1].set_ylabel('True Label', fontsize=11)

# Test Confusion Matrix
cm_test = confusion_matrix(y_true_test_encoded, y_pred_test_encoded)
sns.heatmap(cm_test, annot=True, fmt='d', cmap='Oranges',
            xticklabels=label_encoder.classes_,
            yticklabels=label_encoder.classes_,
            ax=axes[2], cbar_kws={'label': 'Count'})
axes[2].set_title(f'Test Set Confusion Matrix\nAccuracy: {test_metrics_final["Accuracy"]:.4f}',
                  fontsize=13, fontweight='bold')
axes[2].set_xlabel('Predicted Label', fontsize=11)
axes[2].set_ylabel('True Label', fontsize=11)

plt.suptitle('Confusion Matrices - Training, Validation & Test Sets',
             fontsize=16, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

# ==============================
# Classification Reports
# ==============================
print("\n" + "="*60)
print("TRAINING SET - CLASSIFICATION REPORT")
print("="*60)
print(classification_report(y_train_final, train_preds_final,
                          target_names=label_encoder.classes_,
                          digits=4))

print("\n" + "="*60)
print("VALIDATION SET - CLASSIFICATION REPORT")
print("="*60)
print(classification_report(y_val_final, val_preds_final,
                          target_names=label_encoder.classes_,
                          digits=4))

print("\n" + "="*60)
print("TEST SET - CLASSIFICATION REPORT")
print("="*60)
print(classification_report(y_true_test_encoded, y_pred_test_encoded,
                          target_names=label_encoder.classes_,
                          digits=4))

# ==============================
# Per-Class Performance Metrics
# ==============================
print("\n[9/11] Generating per-class performance visualization...")

from sklearn.metrics import precision_recall_fscore_support

# Calculate per-class metrics
train_precision, train_recall, train_f1, _ = precision_recall_fscore_support(
    y_train_final, train_preds_final, average=None, zero_division=0
)
val_precision, val_recall, val_f1, _ = precision_recall_fscore_support(
    y_val_final, val_preds_final, average=None, zero_division=0
)
test_precision, test_recall, test_f1, _ = precision_recall_fscore_support(
    y_true_test_encoded, y_pred_test_encoded, average=None, zero_division=0
)

# Visualization
fig, axes = plt.subplots(1, 3, figsize=(24, 7))
x = np.arange(len(label_encoder.classes_))
width = 0.25

# Precision
axes[0].bar(x - width, train_precision, width, label='Training', alpha=0.8, color='#2ecc71')
axes[0].bar(x, val_precision, width, label='Validation', alpha=0.8, color='#3498db')
axes[0].bar(x + width, test_precision, width, label='Test', alpha=0.8, color='#e74c3c')
axes[0].set_xlabel('Class', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Precision', fontsize=12, fontweight='bold')
axes[0].set_title('Precision per Class', fontsize=14, fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(label_encoder.classes_, rotation=45, ha='right')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3, axis='y')
axes[0].set_ylim([0, 1.1])

# Recall
axes[1].bar(x - width, train_recall, width, label='Training', alpha=0.8, color='#2ecc71')
axes[1].bar(x, val_recall, width, label='Validation', alpha=0.8, color='#3498db')
axes[1].bar(x + width, test_recall, width, label='Test', alpha=0.8, color='#e74c3c')
axes[1].set_xlabel('Class', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Recall', fontsize=12, fontweight='bold')
axes[1].set_title('Recall per Class', fontsize=14, fontweight='bold')
axes[1].set_xticks(x)
axes[1].set_xticklabels(label_encoder.classes_, rotation=45, ha='right')
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3, axis='y')
axes[1].set_ylim([0, 1.1])

# F1-Score
axes[2].bar(x - width, train_f1, width, label='Training', alpha=0.8, color='#2ecc71')
axes[2].bar(x, val_f1, width, label='Validation', alpha=0.8, color='#3498db')
axes[2].bar(x + width, test_f1, width, label='Test', alpha=0.8, color='#e74c3c')
axes[2].set_xlabel('Class', fontsize=12, fontweight='bold')
axes[2].set_ylabel('F1-Score', fontsize=12, fontweight='bold')
axes[2].set_title('F1-Score per Class', fontsize=14, fontweight='bold')
axes[2].set_xticks(x)
axes[2].set_xticklabels(label_encoder.classes_, rotation=45, ha='right')
axes[2].legend(fontsize=11)
axes[2].grid(True, alpha=0.3, axis='y')
axes[2].set_ylim([0, 1.1])

plt.suptitle('Per-Class Performance Metrics - Training, Validation & Test',
             fontsize=16, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

# ==============================
# ROC Curves and AUC Scores
# ==============================
print("\n[10/11] Generating ROC curves...")

# Binarize the labels for ROC curve
y_train_bin = label_binarize(y_train_final, classes=range(len(label_encoder.classes_)))
y_val_bin = label_binarize(y_val_final, classes=range(len(label_encoder.classes_)))
y_test_bin = label_binarize(y_true_test_encoded, classes=range(len(label_encoder.classes_)))

n_classes = len(label_encoder.classes_)

# Compute ROC curve and ROC area for each class
fpr_train = dict()
tpr_train = dict()
roc_auc_train = dict()
fpr_val = dict()
tpr_val = dict()
roc_auc_val = dict()
fpr_test = dict()
tpr_test = dict()
roc_auc_test = dict()

for i in range(n_classes):
    fpr_train[i], tpr_train[i], _ = roc_curve(y_train_bin[:, i], train_preds_proba_final[:, i])
    roc_auc_train[i] = auc(fpr_train[i], tpr_train[i])

    fpr_val[i], tpr_val[i], _ = roc_curve(y_val_bin[:, i], val_preds_proba_final[:, i])
    roc_auc_val[i] = auc(fpr_val[i], tpr_val[i])

    fpr_test[i], tpr_test[i], _ = roc_curve(y_test_bin[:, i], y_pred_proba_test[:, i])
    roc_auc_test[i] = auc(fpr_test[i], tpr_test[i])

# Compute micro-average ROC curve and ROC area
fpr_train["micro"], tpr_train["micro"], _ = roc_curve(y_train_bin.ravel(), train_preds_proba_final.ravel())
roc_auc_train["micro"] = auc(fpr_train["micro"], tpr_train["micro"])

fpr_val["micro"], tpr_val["micro"], _ = roc_curve(y_val_bin.ravel(), val_preds_proba_final.ravel())
roc_auc_val["micro"] = auc(fpr_val["micro"], tpr_val["micro"])

fpr_test["micro"], tpr_test["micro"], _ = roc_curve(y_test_bin.ravel(), y_pred_proba_test.ravel())
roc_auc_test["micro"] = auc(fpr_test["micro"], tpr_test["micro"])

# Plot ROC curves
fig, axes = plt.subplots(1, 3, figsize=(24, 7))

# Colors for classes
colors = cycle(['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd',
                '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22'])

# Training ROC
for i, color in zip(range(n_classes), colors):
    axes[0].plot(fpr_train[i], tpr_train[i], color=color, lw=2,
                label=f'{label_encoder.classes_[i]} (AUC = {roc_auc_train[i]:.3f})')

axes[0].plot(fpr_train["micro"], tpr_train["micro"], color='navy', lw=3, linestyle='--',
            label=f'Micro-avg (AUC = {roc_auc_train["micro"]:.3f})')
axes[0].plot([0, 1], [0, 1], 'k--', lw=2, label='Random Classifier')
axes[0].set_xlim([0.0, 1.0])
axes[0].set_ylim([0.0, 1.05])
axes[0].set_xlabel('False Positive Rate', fontsize=12, fontweight='bold')
axes[0].set_ylabel('True Positive Rate', fontsize=12, fontweight='bold')
axes[0].set_title('ROC Curves - Training Set', fontsize=14, fontweight='bold')
axes[0].legend(loc="lower right", fontsize=9)
axes[0].grid(True, alpha=0.3)

# Validation ROC
colors = cycle(['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd',
                '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22'])

for i, color in zip(range(n_classes), colors):
    axes[1].plot(fpr_val[i], tpr_val[i], color=color, lw=2,
                label=f'{label_encoder.classes_[i]} (AUC = {roc_auc_val[i]:.3f})')

axes[1].plot(fpr_val["micro"], tpr_val["micro"], color='navy', lw=3, linestyle='--',
            label=f'Micro-avg (AUC = {roc_auc_val["micro"]:.3f})')
axes[1].plot([0, 1], [0, 1], 'k--', lw=2, label='Random Classifier')
axes[1].set_xlim([0.0, 1.0])
axes[1].set_ylim([0.0, 1.05])
axes[1].set_xlabel('False Positive Rate', fontsize=12, fontweight='bold')
axes[1].set_ylabel('True Positive Rate', fontsize=12, fontweight='bold')
axes[1].set_title('ROC Curves - Validation Set', fontsize=14, fontweight='bold')
axes[1].legend(loc="lower right", fontsize=9)
axes[1].grid(True, alpha=0.3)

# Test ROC
colors = cycle(['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd',
                '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22'])

for i, color in zip(range(n_classes), colors):
    axes[2].plot(fpr_test[i], tpr_test[i], color=color, lw=2,
                label=f'{label_encoder.classes_[i]} (AUC = {roc_auc_test[i]:.3f})')

axes[2].plot(fpr_test["micro"], tpr_test["micro"], color='navy', lw=3, linestyle='--',
            label=f'Micro-avg (AUC = {roc_auc_test["micro"]:.3f})')
axes[2].plot([0, 1], [0, 1], 'k--', lw=2, label='Random Classifier')
axes[2].set_xlim([0.0, 1.0])
axes[2].set_ylim([0.0, 1.05])
axes[2].set_xlabel('False Positive Rate', fontsize=12, fontweight='bold')
axes[2].set_ylabel('True Positive Rate', fontsize=12, fontweight='bold')
axes[2].set_title('ROC Curves - Test Set', fontsize=14, fontweight='bold')
axes[2].legend(loc="lower right", fontsize=9)
axes[2].grid(True, alpha=0.3)

plt.suptitle('Multi-class ROC Curves with AUC Scores - All Sets',
             fontsize=16, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

# Print AUC scores
print("\n" + "="*80)
print("PER-CLASS AUC SCORES")
print("="*80)
print(f"{'Class':<15} {'Training AUC':<18} {'Validation AUC':<18} {'Test AUC':<18}")
print("-"*80)
for i, class_name in enumerate(label_encoder.classes_):
    print(f"{class_name:<15} {roc_auc_train[i]:<18.4f} {roc_auc_val[i]:<18.4f} {roc_auc_test[i]:<18.4f}")
print("-"*80)
print(f"{'Micro-average':<15} {roc_auc_train['micro']:<18.4f} {roc_auc_val['micro']:<18.4f} {roc_auc_test['micro']:<18.4f}")
print("="*80)

# ==============================
# Sample Test Predictions
# ==============================
print(f"\n[11/11] Displaying {FINAL_CONFIG['n_predictions_to_show']} sample predictions from test set...")

# Select random samples
sample_indices = random.sample(range(len(test_image_info_final)),
                              min(FINAL_CONFIG['n_predictions_to_show'], len(test_image_info_final)))

# Create visualization grid
n_cols = 5
n_rows = (len(sample_indices) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 4*n_rows))
axes = axes.ravel() if n_rows > 1 else [axes] if n_cols == 1 else axes

for idx, sample_idx in enumerate(sample_indices):
    sample = test_image_info_final[sample_idx]

    # Load and display image
    img = cv2.imread(sample['path'])
    if img is not None:
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        axes[idx].imshow(img_rgb)

        # Determine if prediction is correct
        is_correct = sample['true_label'] == sample['pred_label']
        color = 'green' if is_correct else 'red'
        status = '✓' if is_correct else '✗'

        title = f"{status} True: {sample['true_label']}\n"
        title += f"Pred: {sample['pred_label']}\n"
        title += f"Conf: {sample['confidence']:.3f}"

        axes[idx].set_title(title, fontsize=10, fontweight='bold', color=color)

    axes[idx].axis('off')

# Hide unused subplots
for idx in range(len(sample_indices), len(axes)):
    axes[idx].axis('off')

plt.suptitle('Sample Predictions from Test Set', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

# Print prediction summary
correct_predictions = sum(1 for info in test_image_info_final
                         if info['true_label'] == info['pred_label'])
total_predictions = len(test_image_info_final)
print(f"\nTest Set Summary:")
print(f"  Correct predictions: {correct_predictions}/{total_predictions}")
print(f"  Accuracy: {correct_predictions/total_predictions:.4f}")
# ==============================
# 🔍 GLOBAL SHAP EXPLANATIONS (FIXED)
# ==============================
print("\n" + "="*80)
print("GENERATING GLOBAL SHAP EXPLANATIONS FOR FINAL MODEL")
print("="*80)

# Sample data for SHAP analysis
shap_sample_size = min(FINAL_CONFIG['shap_samples'], len(X_pca))
shap_sample_indices = np.random.choice(len(X_pca), shap_sample_size, replace=False)
X_shap_sample = X_pca[shap_sample_indices]

# Generate feature names for PCA-based features
feature_names = [f"PC_{i+1}" for i in range(X_pca.shape[1])]

# Try DeepExplainer first, fallback to KernelExplainer
try:
    print("→ Trying DeepExplainer...")
    explainer = shap.DeepExplainer(final_model, X_shap_sample[:50])
    shap_values = explainer.shap_values(X_shap_sample)
    print("✓ DeepExplainer succeeded.")
except Exception as e:
    print(f"⚠️ DeepExplainer failed: {e}")
    print("→ Falling back to KernelExplainer (slower but more stable)...")
    explainer = shap.KernelExplainer(final_model.predict, X_shap_sample[:30])
    shap_values = explainer.shap_values(X_shap_sample[:100])
    print("✓ KernelExplainer succeeded.")

# ==============================
# 🧠 Verify SHAP output structure
# ==============================
print("\nChecking SHAP output structure...")
if isinstance(shap_values, list):
    print(f"SHAP returned a list with {len(shap_values)} arrays.")
    print(f"Each array shape: {shap_values[0].shape}")
else:
    print(f"SHAP returned a single array with shape: {shap_values.shape}")

# ==============================
# 📊 Global SHAP Summary Plots
# ==============================
print("\nGenerating global SHAP summary plots...")

plt.figure(figsize=(12, 6))
shap.summary_plot(
    shap_values[0] if isinstance(shap_values, list) else shap_values,
    X_shap_sample,
    feature_names=feature_names,
    show=False
)
# plt.title("Global SHAP Summary Plot (Feature Impact on Predictions)", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 5))
shap.summary_plot(
    shap_values[0] if isinstance(shap_values, list) else shap_values,
    X_shap_sample,
    feature_names=feature_names,
    plot_type="bar",
    show=False
)
# plt.title("Global SHAP Feature Importance (Mean |SHAP| Value)", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# ==============================
# 🎯 Per-Class SHAP Plots (only if multi-class)
# ==============================
if isinstance(shap_values, list) and len(shap_values) == len(label_encoder.classes_):
    print("\nGenerating per-class SHAP summary plots...")
    for i, class_name in enumerate(label_encoder.classes_):
        print(f"→ Class {i}: {class_name}")
        plt.figure(figsize=(12, 6))
        shap.summary_plot(shap_values[i], X_shap_sample, feature_names=feature_names, show=False)
        plt.title(f"SHAP Summary Plot for Class: {class_name}", fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.show()
else:
    print("\nSkipping per-class SHAP plots (model outputs not multi-class structured).")

# ==============================
# 💾 Save SHAP Results
# ==============================
np.save('final_shap_values.npy', shap_values)
np.save('final_shap_sample.npy', X_shap_sample)
print("✓ SHAP values and sample data saved.")



# ==============================
# Save Final Model
# ==============================
print("\n" + "="*80)
print("SAVING FINAL MODEL AND RESULTS")
print("="*80)

try:
    # Save model
    final_model.save('final_model.h5')
    print("✓ Model saved as 'final_model.h5'")

    # Save metrics
    final_results = {
        'Model': 'Final Model',
        **{f'Val_{k}': v for k, v in val_metrics_final.items()},
        **{f'Test_{k}': v for k, v in test_metrics_final.items()}
    }

    pd.DataFrame([final_results]).to_csv('final_model_metrics.csv', index=False)
    print("✓ Metrics saved as 'final_model_metrics.csv'")

    # Save test predictions
    test_results_df = pd.DataFrame(test_image_info_final)
    test_results_df.to_csv('final_test_predictions.csv', index=False)
    print("✓ Test predictions saved as 'final_test_predictions.csv'")

    # Save per-class metrics
    per_class_metrics = pd.DataFrame({
        'Class': label_encoder.classes_,
        'Val_Precision': val_precision,
        'Val_Recall': val_recall,
        'Val_F1': val_f1,
        'Val_AUC': [roc_auc_val[i] for i in range(n_classes)],
        'Test_Precision': test_precision,
        'Test_Recall': test_recall,
        'Test_F1': test_f1,
        'Test_AUC': [roc_auc_test[i] for i in range(n_classes)]
    })
    per_class_metrics.to_csv('final_per_class_metrics.csv', index=False)
    print("✓ Per-class metrics saved as 'final_per_class_metrics.csv'")

except Exception as e:
    print(f"Error saving results: {e}")

# ==============================
# Final Summary
# ==============================
print("\n" + "="*80)
print("FINAL MODEL EVALUATION COMPLETE!")
print("="*80)
print("\n📊 SUMMARY:")
print(f"  • Training Samples: {len(X_train_final)}")
print(f"  • Validation Samples: {len(X_val_final)}")
print(f"  • Test Samples: {len(test_image_info_final)}")
print(f"  • Number of Classes: {len(label_encoder.classes_)}")
print(f"\n🎯 VALIDATION PERFORMANCE:")
print(f"  • Accuracy: {val_metrics_final['Accuracy']:.4f}")
print(f"  • F1-Score: {val_metrics_final['F1']:.4f}")
print(f"  • Cohen's Kappa: {val_metrics_final['Cohen_Kappa']:.4f}")
print(f"  • MCC: {val_metrics_final['MCC']:.4f}")
print(f"  • AUC: {val_metrics_final['AUC']:.4f}")
print(f"\n🎯 TEST PERFORMANCE:")
print(f"  • Accuracy: {test_metrics_final['Accuracy']:.4f}")
print(f"  • F1-Score: {test_metrics_final['F1']:.4f}")
print(f"  • Cohen's Kappa: {test_metrics_final['Cohen_Kappa']:.4f}")
print(f"  • MCC: {test_metrics_final['MCC']:.4f}")
print(f"  • AUC: {test_metrics_final['AUC']:.4f}")
print("\n✓ All visualizations generated")
print("✓ LIME explanations completed")
# print("✓ SHAP explanations completed")
print("✓ Model and results saved")
print("="*80)

In [ ]:
# ==============================
# Confusion Matrix - TEST SET ONLY
# ==============================
print("\n[8/11] Generating test set confusion matrix...")

fig, ax = plt.subplots(figsize=(10, 8))

# Test Confusion Matrix
cm_test = confusion_matrix(y_true_test_encoded, y_pred_test_encoded)
sns.heatmap(cm_test, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_encoder.classes_,
            yticklabels=label_encoder.classes_,
            ax=ax, cbar_kws={'label': 'Count'})
# ax.set_title(f'Test Set Confusion Matrix\nAccuracy: {test_metrics_final["Accuracy"]:.4f}', fontsize=14, fontweight='bold')
ax.set_xlabel('Predicted Label', fontsize=12)
ax.set_ylabel('True Label', fontsize=12)

plt.tight_layout()
plt.show()

# ==============================
# Classification Report - TEST SET
# ==============================
print("\n" + "="*60)
print("TEST SET - CLASSIFICATION REPORT")
print("="*60)
print(classification_report(y_true_test_encoded, y_pred_test_encoded,
                          target_names=label_encoder.classes_,
                          digits=4))

# ==============================
# Per-Class Performance Metrics - TEST SET ONLY
# ==============================
print("\n[9/11] Generating test set per-class performance visualization...")

from sklearn.metrics import precision_recall_fscore_support

# Calculate per-class metrics for test set
test_precision, test_recall, test_f1, _ = precision_recall_fscore_support(
    y_true_test_encoded, y_pred_test_encoded, average=None, zero_division=0
)

# Visualization
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
x = np.arange(len(label_encoder.classes_))
width = 0.5

# Precision
axes[0].bar(x, test_precision, width, alpha=0.8, color='#e74c3c')
axes[0].set_xlabel('Class', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Precision', fontsize=12, fontweight='bold')
axes[0].set_title('Precision per Class - Test Set', fontsize=14, fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(label_encoder.classes_, rotation=45, ha='right')
axes[0].grid(True, alpha=0.3, axis='y')
axes[0].set_ylim([0, 1.1])

# Add value labels on bars
for i, v in enumerate(test_precision):
    axes[0].text(i, v + 0.02, f'{v:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

# Recall
axes[1].bar(x, test_recall, width, alpha=0.8, color='#e74c3c')
axes[1].set_xlabel('Class', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Recall', fontsize=12, fontweight='bold')
axes[1].set_title('Recall per Class - Test Set', fontsize=14, fontweight='bold')
axes[1].set_xticks(x)
axes[1].set_xticklabels(label_encoder.classes_, rotation=45, ha='right')
axes[1].grid(True, alpha=0.3, axis='y')
axes[1].set_ylim([0, 1.1])

# Add value labels on bars
for i, v in enumerate(test_recall):
    axes[1].text(i, v + 0.02, f'{v:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

# F1-Score
axes[2].bar(x, test_f1, width, alpha=0.8, color='#e74c3c')
axes[2].set_xlabel('Class', fontsize=12, fontweight='bold')
axes[2].set_ylabel('F1-Score', fontsize=12, fontweight='bold')
axes[2].set_title('F1-Score per Class - Test Set', fontsize=14, fontweight='bold')
axes[2].set_xticks(x)
axes[2].set_xticklabels(label_encoder.classes_, rotation=45, ha='right')
axes[2].grid(True, alpha=0.3, axis='y')
axes[2].set_ylim([0, 1.1])

# Add value labels on bars
for i, v in enumerate(test_f1):
    axes[2].text(i, v + 0.02, f'{v:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.suptitle('Per-Class Performance Metrics - Test Set',
             fontsize=16, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

# ==============================
# ROC Curve - TEST SET ONLY
# ==============================
print("\n[10/11] Generating test set ROC curve...")

# Binarize the labels for ROC curve
y_test_bin = label_binarize(y_true_test_encoded, classes=range(len(label_encoder.classes_)))
n_classes = len(label_encoder.classes_)

# Compute ROC curve and ROC area for each class
fpr_test = dict()
tpr_test = dict()
roc_auc_test = dict()

for i in range(n_classes):
    fpr_test[i], tpr_test[i], _ = roc_curve(y_test_bin[:, i], y_pred_proba_test[:, i])
    roc_auc_test[i] = auc(fpr_test[i], tpr_test[i])

# Compute micro-average ROC curve and ROC area
fpr_test["micro"], tpr_test["micro"], _ = roc_curve(y_test_bin.ravel(), y_pred_proba_test.ravel())
roc_auc_test["micro"] = auc(fpr_test["micro"], tpr_test["micro"])

# Plot ROC curve
fig, ax = plt.subplots(figsize=(12, 10))

# Colors for classes
colors = cycle(['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd',
                '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22'])

# Plot ROC curve for each class
for i, color in zip(range(n_classes), colors):
    ax.plot(fpr_test[i], tpr_test[i], color=color, lw=2.5,
            label=f'{label_encoder.classes_[i]} (AUC = {roc_auc_test[i]:.3f})')

# Plot micro-average ROC curve
ax.plot(fpr_test["micro"], tpr_test["micro"], color='navy', lw=3.5, linestyle='--',
        label=f'Micro-average (AUC = {roc_auc_test["micro"]:.3f})')

# Plot random classifier line
ax.plot([0, 1], [0, 1], 'k--', lw=2, label='Random Classifier')

ax.set_xlim([0.0, 0.3])
ax.set_ylim([0.7, 1.0])
ax.set_xlabel('False Positive Rate', fontsize=13, fontweight='bold')
ax.set_ylabel('True Positive Rate', fontsize=13, fontweight='bold')
# ax.set_title('ROC Curves - Test Set', fontsize=15, fontweight='bold', pad=15)
# ax.legend(loc="lower right", fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()



# Print AUC scores
print("\n" + "="*60)
print("TEST SET - PER-CLASS AUC SCORES")
print("="*60)
print(f"{'Class':<20} {'AUC Score':<15}")
print("-"*60)
for i, class_name in enumerate(label_encoder.classes_):
    print(f"{class_name:<20} {roc_auc_test[i]:<15.4f}")
print("-"*60)
print(f"{'Micro-average':<20} {roc_auc_test['micro']:<15.4f}")
print("="*60)

In [ ]:
print(model.optimizer)
# print(train_dir)